# **ENSO Growth/Decay Rate Experiment - Models**

Runs Both: 

(1) Unconditional (all eruptions)

(2) Phase-conditioned (eruption-year phase = El Niño / Neutral / La Niña)

Output: 

-  CSV saved to ~/ENSO-GDR-Models.csv

In [1]:
import os
import numpy as np
import pandas as pd
import xarray as xr
import cftime

# File Paths
model_paths = {
    "ACCESS-ESM1.5":      "/g/data/ob22/jxb548/PMIPDATA/past1000/ts_ACCESS-ESM1.5_r1i1p1_0850_1849_r240x120.nc",
    "BCC-CSM1-1":         "/g/data/ob22/jxb548/PMIPDATA/past1000/ts_bcc-csm1-1_r1i1p1_0850_2000_r240x120.nc",
    "CCSM4":              "/g/data/ob22/jxb548/PMIPDATA/past1000/ts_CCSM4_r1i1p1_0850_1850_r240x120.nc",
    "CSIRO-Mk3L-1-2":     "/g/data/ob22/jxb548/PMIPDATA/past1000/ts_CSIRO-Mk3L-1-2_r1i1p1_0851_1850_r240x120.nc",
    "GISS-E2-R":          "/g/data/ob22/jxb548/PMIPDATA/past1000/ts_GISS-E2-R_r1i1p121_0850_1850_r240x120.nc",
    "IPSL-CM5A-LR":       "/g/data/ob22/jxb548/PMIPDATA/past1000/ts_IPSL-CM5A-LR_r1i1p1_0850_1850_r240x120.nc",
    "MIROC-ES2L":         "/g/data/ob22/jxb548/PMIPDATA/past1000/ts_MIROC-ES2L_r1i1p1_0850_1849_r240x120.nc",
    "MIROC-ESM":          "/g/data/ob22/jxb548/PMIPDATA/past1000/ts_MIROC-ESM_r1i1p1_0850_1849_r240x120.nc",
    "MPI-ESM-P":          "/g/data/ob22/jxb548/PMIPDATA/past1000/ts_MPI-ESM-P_r1i1p1_0850_1849_r240x120.nc",
}

# Eruption List with known seasonality
ERUPTIONS_RAW = [
    (939.0,  4.0,  1.0,  63.6, -1.0, 16.23, 4.97),
    (946.0, 11.0,  1.0,  42.0, -1.0,  1.72, 0.61),
    (1257.0, 7.0,  1.0,  -8.4,  1.4, 59.42, 10.86),
    (1477.0, 2.0,  1.0,  64.6, -1.0,  5.12, 1.61),
    (1510.0, 7.0, 25.0,  64.0, -1.0,  2.30, 0.83),
    (1585.0, 1.0, 10.0,  19.5, 10.6,  8.51, 2.34),
    (1595.0, 3.0,  1.0,   4.9,  0.8,  8.87, 1.51),
    (1600.0, 2.0, 17.0, -16.6,  2.0, 18.95, 4.03),
    (1640.0, 12.0, 26.0,  6.1,  2.8, 18.68, 4.28),
    (1667.0, 9.0, 23.0,  42.7, -1.0,  3.48, 1.11),
    (1673.0, 5.0, 20.0,   1.4,  0.7,  4.67, 0.82),
    (1707.0, 12.0, 16.0,  35.4, -1.0,  1.08, 0.40),
    (1721.0, 5.0, 11.0,  63.6, -1.0,  0.81, 0.36),
    (1739.0, 8.0, 19.0,  42.7, -1.0,  3.44, 1.09),
    (1755.0, 10.0, 17.0,  63.6, -1.0,  1.18, 0.43),
    (1766.0, 4.0,  5.0,  64.0, -1.0,  2.52, 0.75),
    (1783.0, 6.0, 15.0,  64.4, -1.0, 20.81, 7.04),
    (1815.0, 4.0, 10.0,  -8.0,  0.8, 28.08, 4.49),
    (1822.0, 10.0,  8.0,  -7.3, 10.0,  2.02, 0.79),
    (1835.0, 1.0, 20.0,  13.0,  2.0,  9.48, 2.21),
]
eruptions_df = pd.DataFrame(
    ERUPTIONS_RAW,
    columns=["yearCE", "month", "day", "lat", "hemi", "ssi", "sigma_ssi"],
)


ERUPTION_YEARS = eruptions_df["yearCE"].to_numpy(dtype=int)

In [2]:
# Settings

LAT_MIN, LAT_MAX = -5.0, 5.0
LON_MIN, LON_MAX = 190.0, 240.0
 
TROP_LAT_MIN, TROP_LAT_MAX = -20.0, 20.0
 
THRESHOLD = 0.5
PHASES = ["El Niño", "Neutral", "La Niña"]
 
LAG_MIN, LAG_MAX = -5, 5
FIT_LAG_MIN, FIT_LAG_MAX = 0, 3
 
EXCLUDE_MARGIN_YEARS = 5
N_MC = 4000
RNG_SEED = 42
MIN_EVENTS = 3
 
EPS = 1e-6
USE_ROBUST_CENTER = True
 
# OUTPUT SAVING
SAVE_CSV = True
OUT_DIR  = "/home/563/ft3359/FT-Honours/Honours_Paper/GDR/All_Eruptions"
OUT_NAME = "ENSO-GDR-Models.csv"

In [3]:
# Pipeline Functions

def load_sst_data(file_path):
    ds = xr.open_dataset(file_path, use_cftime=True)
    sst = ds["ts"]
 
    if float(sst.max()) > 200:
        sst = sst - 273.15
 
    t0 = sst["time"].values[0]
    start_str = "0850-01-01"
    end_str   = "1849-12-31"
    if isinstance(t0, cftime.Datetime360Day):
        end_str = "1849-12-30"
 
    sst = sst.sel(time=slice(start_str, end_str))
    return sst.chunk({"time": 365})
 
 
def area_weighted_mean_latlon(da: xr.DataArray) -> xr.DataArray:

    w = np.cos(np.deg2rad(da["lat"]))
    w = xr.DataArray(w, coords={"lat": da["lat"]}, dims=("lat",))
    return da.weighted(w).mean(dim=["lat", "lon"])
 
 
def calculate_relative_nino34_monthly(sst_1000):

    n34 = sst_1000.sel(lat=slice(LAT_MIN, LAT_MAX), lon=slice(LON_MIN, LON_MAX)).mean(dim=["lat", "lon"])
 
    trop = area_weighted_mean_latlon(sst_1000.sel(lat=slice(TROP_LAT_MIN, TROP_LAT_MAX)))
 
    n34_anom  = n34.groupby("time.month")  - n34.groupby("time.month").mean("time")
    trop_anom = trop.groupby("time.month") - trop.groupby("time.month").mean("time")
 
    rel = n34_anom - trop_anom
 
    return rel.rolling(time=3, center=True).mean()
 
 
def calculate_juljun_annual(monthly_3mo):
    t = monthly_3mo["time"]
    enso_year = xr.where(t.dt.month >= 7, t.dt.year, t.dt.year - 1)
 
    annual = monthly_3mo.groupby(enso_year).mean("time")
 
    years = annual["group"].values.astype(int)
    annual = annual.rename({"group": "time"}).assign_coords(time=years)
    return annual
 
 
def classify_phase_value(v, thr=0.5):
    if not np.isfinite(v):
        return None
    if v >= thr:
        return "El Niño"
    if v <= -thr:
        return "La Niña"
    return "Neutral"
 
 
# Rate Experiment Helpers
 
def in_eruption_margin_fast(years: np.ndarray, eruption_years: np.ndarray, margin: int) -> np.ndarray:
    years = years.astype(int)
    e_sorted = np.sort(eruption_years.astype(int))
    lo = np.searchsorted(e_sorted, years - margin, side="left")
    hi = np.searchsorted(e_sorted, years + margin, side="right")
    return (hi > lo)
 
 
def extract_event_windows_from_annual(years: np.ndarray, values: np.ndarray,
                                      event_years: np.ndarray, lag_min: int, lag_max: int) -> np.ndarray:
    y_to_i = {int(y): i for i, y in enumerate(years)}
    lags = np.arange(lag_min, lag_max + 1, dtype=int)
    mat = np.full((len(event_years), len(lags)), np.nan, dtype=float)
    for ei, ey in enumerate(event_years):
        ey = int(ey)
        for li, lag in enumerate(lags):
            yy = ey + int(lag)
            if yy in y_to_i:
                mat[ei, li] = values[y_to_i[yy]]
    return mat
 
 
def composite_mean(mat: np.ndarray) -> np.ndarray:
    return np.nanmean(mat, axis=0)
 
 
def fit_exponential_rate(t: np.ndarray, x: np.ndarray) -> float:
    t = np.asarray(t, float)
    x = np.asarray(x, float)
    ok = np.isfinite(t) & np.isfinite(x)
    t, x = t[ok], x[ok]
    if t.size < 3:
        return np.nan
 
    C = float(np.median(x[-2:])) if x.size >= 2 else float(x[-1])
    y = np.abs(x - C)
    good = np.isfinite(y) & (y > EPS)
    t2, y2 = t[good], y[good]
    if t2.size < 3:
        return np.nan
 
    logy = np.log(y2)
    A = np.vstack([t2, np.ones_like(t2)]).T
    slope, _ = np.linalg.lstsq(A, logy, rcond=None)[0]
    return float(slope)
 
 
def two_sided_p(null: np.ndarray, obs: float) -> float:
    null = np.asarray(null, float)
    null = null[np.isfinite(null)]
    if null.size == 0 or (not np.isfinite(obs)):
        return np.nan
 
    if USE_ROBUST_CENTER:
        c = float(np.median(null))
        dev = abs(obs - c)
        return float(np.mean(np.abs(null - c) >= dev))
    else:
        return float(np.mean(np.abs(null) >= abs(obs)))
 
 
def pick_control_years_unconditional(years: np.ndarray,
                                     eruption_years: np.ndarray,
                                     exclude_margin: int,
                                     n: int,
                                     rng: np.random.Generator) -> np.ndarray:
    years = np.asarray(years, int)
    in_margin = in_eruption_margin_fast(years, eruption_years, exclude_margin)
    ok = (~in_margin) & (~np.isin(years, eruption_years))
    pool = years[ok]
    if pool.size == 0:
        return np.array([], dtype=int)
    replace = pool.size < n
    return rng.choice(pool, size=n, replace=replace).astype(int)
 
 
def pick_control_years_phase_matched(years: np.ndarray,
                                     phase_by_year: dict,
                                     target_phase: str,
                                     eruption_years: np.ndarray,
                                     exclude_margin: int,
                                     n: int,
                                     rng: np.random.Generator) -> np.ndarray:
    years = np.asarray(years, int)
    in_margin = in_eruption_margin_fast(years, eruption_years, exclude_margin)
    ok_years = years[(~in_margin) & (~np.isin(years, eruption_years))]
 
    pool = np.array([y for y in ok_years if phase_by_year.get(int(y), None) == target_phase], dtype=int)
    if pool.size == 0:
        return np.array([], dtype=int)
    replace = pool.size < n
    return rng.choice(pool, size=n, replace=replace).astype(int)
 
 
def case_to_simple(case: str) -> str:
    if case == "Unconditional":
        return "Unconditional"
    if isinstance(case, str) and case.startswith("Phase-conditioned:"):
        return case.split("Phase-conditioned:", 1)[1].strip()
    return case

In [4]:
# Main
 
lags = np.arange(LAG_MIN, LAG_MAX + 1, dtype=int)
fit_mask = (lags >= FIT_LAG_MIN) & (lags <= FIT_LAG_MAX)
t_fit = lags[fit_mask].astype(float)
 
rows = []
 
for model_name, path in model_paths.items():
    print(f"\nLoading {model_name}...")
    try:
        sst = load_sst_data(path)
 
        rel_monthly = calculate_relative_nino34_monthly(sst)
        rel_annual_c = calculate_juljun_annual(rel_monthly)
 
    except Exception as e:
        print(f"  Failed {model_name}: {type(e).__name__}: {e}")
        continue
 
    years_avail = rel_annual_c.time.values.astype(int)
    vals_annual = rel_annual_c.values.astype(float)
    years_set = set(int(y) for y in years_avail)
 
    erupt_years = np.array([y for y in ERUPTION_YEARS if int(y) in years_set], dtype=int)
 
    phase_by_year = {}
    for y, v in zip(years_avail, vals_annual):
        ph = classify_phase_value(float(v), THRESHOLD)
        if ph is not None:
            phase_by_year[int(y)] = ph
 
    ev_by_phase = {ph: [] for ph in PHASES}
    for ey in erupt_years:
        ph = phase_by_year.get(int(ey), None)
        if ph in ev_by_phase:
            ev_by_phase[ph].append(int(ey))
    ev_by_phase = {k: np.array(v, dtype=int) for k, v in ev_by_phase.items()}
 
    rng = np.random.default_rng(RNG_SEED + (abs(hash(model_name)) % 100_000))
 
    # Unconditional
    
    ev = erupt_years.copy()
    if ev.size < MIN_EVENTS:
        rows.append(dict(dataset=model_name, case="Unconditional", case_simple="Unconditional",
                         N_events=int(ev.size), r_yr1=np.nan, tau_yr=np.nan, p_value=np.nan))
    else:
        matE = extract_event_windows_from_annual(years_avail, vals_annual, ev, LAG_MIN, LAG_MAX)
        compE = composite_mean(matE)
        r_obs = fit_exponential_rate(t_fit, compE[fit_mask])
        tau = (-1.0 / r_obs) if (np.isfinite(r_obs) and r_obs < 0) else np.nan
 
        r_null = np.full(N_MC, np.nan, dtype=float)
        for i in range(N_MC):
            ctrl = pick_control_years_unconditional(years_avail, erupt_years, EXCLUDE_MARGIN_YEARS, ev.size, rng)
            if ctrl.size == 0:
                continue
            matC = extract_event_windows_from_annual(years_avail, vals_annual, ctrl, LAG_MIN, LAG_MAX)
            compC = composite_mean(matC)
            r_null[i] = fit_exponential_rate(t_fit, compC[fit_mask])
 
        p_mc = two_sided_p(r_null, r_obs)
 
        rows.append(dict(dataset=model_name, case="Unconditional", case_simple="Unconditional",
                         N_events=int(ev.size),
                         r_yr1=float(r_obs) if np.isfinite(r_obs) else np.nan,
                         tau_yr=float(tau) if np.isfinite(tau) else np.nan,
                         p_value=float(p_mc) if np.isfinite(p_mc) else np.nan))
 
    # Phase-conditioned
    
    for ph in PHASES:
        evp = ev_by_phase[ph]
        if evp.size < MIN_EVENTS:
            rows.append(dict(dataset=model_name, case=f"Phase-conditioned: {ph}", case_simple=ph,
                             N_events=int(evp.size), r_yr1=np.nan, tau_yr=np.nan, p_value=np.nan))
            continue
 
        matE = extract_event_windows_from_annual(years_avail, vals_annual, evp, LAG_MIN, LAG_MAX)
        compE = composite_mean(matE)
        r_obs = fit_exponential_rate(t_fit, compE[fit_mask])
        tau = (-1.0 / r_obs) if (np.isfinite(r_obs) and r_obs < 0) else np.nan
 
        r_null = np.full(N_MC, np.nan, dtype=float)
        for i in range(N_MC):
            ctrl = pick_control_years_phase_matched(
                years_avail, phase_by_year, ph, erupt_years, EXCLUDE_MARGIN_YEARS, evp.size, rng
            )
            if ctrl.size == 0:
                continue
            matC = extract_event_windows_from_annual(years_avail, vals_annual, ctrl, LAG_MIN, LAG_MAX)
            compC = composite_mean(matC)
            r_null[i] = fit_exponential_rate(t_fit, compC[fit_mask])
 
        p_mc = two_sided_p(r_null, r_obs)
 
        rows.append(dict(dataset=model_name, case=f"Phase-conditioned: {ph}", case_simple=ph,
                         N_events=int(evp.size),
                         r_yr1=float(r_obs) if np.isfinite(r_obs) else np.nan,
                         tau_yr=float(tau) if np.isfinite(tau) else np.nan,
                         p_value=float(p_mc) if np.isfinite(p_mc) else np.nan))
 
 
out = pd.DataFrame(rows)
 
print(out[["dataset","case","N_events","r_yr1","tau_yr","p_value"]].to_string(index=False))


Loading ACCESS-ESM1.5...


/g/data/xp65/admin/analysis3/sitecustomize.py:72: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  mod = _real_import(name, globals, locals, fromlist, level)
/jobfs/173579859.gadi-pbs/ipykernel_1915720/493707179.py:4: DeprecationWarning: Usage of 'use_cftime' as a kwarg is deprecated. Please pass a 'CFDatetimeCoder' instance initialized with 'use_cftime' to the 'decode_times' kwarg instead.
Example usage:
    time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
    ds = xr.open_dataset(decode_times=time_coder)

  ds = xr.open_dataset(file_path, use_cftime=True)



Loading BCC-CSM1-1...


/jobfs/173579859.gadi-pbs/ipykernel_1915720/493707179.py:4: DeprecationWarning: Usage of 'use_cftime' as a kwarg is deprecated. Please pass a 'CFDatetimeCoder' instance initialized with 'use_cftime' to the 'decode_times' kwarg instead.
Example usage:
    time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
    ds = xr.open_dataset(decode_times=time_coder)

  ds = xr.open_dataset(file_path, use_cftime=True)



Loading CCSM4...


...


Loading MIROC-ESM...


/jobfs/173579859.gadi-pbs/ipykernel_1915720/493707179.py:4: DeprecationWarning: Usage of 'use_cftime' as a kwarg is deprecated. Please pass a 'CFDatetimeCoder' instance initialized with 'use_cftime' to the 'decode_times' kwarg instead.
Example usage:
    time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
    ds = xr.open_dataset(decode_times=time_coder)

  ds = xr.open_dataset(file_path, use_cftime=True)



Loading MPI-ESM-P...


/jobfs/173579859.gadi-pbs/ipykernel_1915720/493707179.py:4: DeprecationWarning: Usage of 'use_cftime' as a kwarg is deprecated. Please pass a 'CFDatetimeCoder' instance initialized with 'use_cftime' to the 'decode_times' kwarg instead.
Example usage:
    time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
    ds = xr.open_dataset(decode_times=time_coder)

  ds = xr.open_dataset(file_path, use_cftime=True)


       dataset                       case  N_events     r_yr1   tau_yr  p_value
 ACCESS-ESM1.5              Unconditional        20 -0.101663 9.836385  0.83725
 ACCESS-ESM1.5 Phase-conditioned: El Niño         3 -0.376871 2.653427  0.81550
 ACCESS-ESM1.5 Phase-conditioned: Neutral        14  0.064334      NaN  0.77925
 ACCESS-ESM1.5 Phase-conditioned: La Niña         3 -0.376400 2.656751  0.99150
    BCC-CSM1-1              Unconditional        20  0.196718      NaN  0.56725
    BCC-CSM1-1 Phase-conditioned: El Niño         5 -0.288923 3.461135  0.31600
    BCC-CSM1-1 Phase-conditioned: Neutral        15  0.520416      NaN  0.28182
    BCC-CSM1-1 Phase-conditioned: La Niña         0       NaN      NaN      NaN
         CCSM4              Unconditional        20 -0.919012 1.088125  0.19575
         CCSM4 Phase-conditioned: El Niño         8 -0.571945 1.748419  0.47425
         CCSM4 Phase-conditioned: Neutral         9 -0.379138 2.637562  0.52275
         CCSM4 Phase-conditioned: La Niñ

In [5]:
# Save 

if SAVE_CSV: 
    os.makedirs(OUT_DIR, exist_ok=True)
    out_path = os.path.join(OUT_DIR, OUT_NAME)
    out[["dataset", "case_simple", "N_events", "r_yr1", "tau_yr", "p_value"]].to_csv(out_path, index=False)
    print("\nSaved CSV:", out_path)


Saved CSV: /home/563/ft3359/FT-Honours/Honours_Paper/GDR/All_Eruptions/ENSO-GDR-Models.csv
